In [73]:
import optbinning
import xgboost as xgb
from xgboost import XGBClassifier
from optbinning import OptimalBinning
import pickle
import pandas as pd
import numpy as np
import os as os
import gc
gc.collect()
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from xgboost import plot_tree
from tqdm import tqdm
import sklearn
from sklearn import linear_model
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

In [74]:
df = pd.read_excel("fin_data_of_proj.xlsx")

In [75]:
# Unfin_cost and Season of every year

def get_season(date):
    month = date.month if pd.notnull(date) else None
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Fall'
    else:
        return None

df['season'] = df['dpg_date'].apply(get_season)
df['unfin_coff'] = df['unfin_cost']/df['cost_plan']

In [76]:
# Inflaction of every year 8%

annual_inflation_rate = 0.08
inflation_reference_date = datetime(2024, 11, 1)


df['dpg_date'] = pd.to_datetime(df['dpg_date'], errors='coerce')

financial_columns = [
    'cost_plan', 'flat_msprice', 'build_mscost', 'flat_mscost', 'com_msprice', 'park_price',
    'self_cost', 'unfin_cost', 'share_cost', 'warr_cost', 'comm_cost', 'cost_official',
    'Итого краткосрочные активы', 'Итого краткосрочных обязательств', 'Денежные средства',
    'Краткосрочная дебиторская задолженность', 'Запасы', 'Себестоимость', 'Доход от реализации',
    'Итого активы', 'Итого капитал', 'Итого пассивы', 'Краткосрочная кредиторская задолженность',
    'Операционная прибыль', 'Долгосрочные финансовые обязательства', 'Краткосрочные финансовые обязательства',
    'Итоговая прибыль (убыток)', 'Расходы на финансирование'
]

df['years_since_dpg'] = (inflation_reference_date - df['dpg_date']).dt.days / 365.25
df['inflation_factor'] = (1 + annual_inflation_rate) ** df['years_since_dpg']

for col in financial_columns:
    if col in df.columns:
        df[col] = df[col] * df['inflation_factor']

df.drop(columns=['years_since_dpg', 'inflation_factor'], inplace=True)


In [77]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

df.replace([np.inf, -np.inf], np.nan, inplace=True)

df = df[:-11]
df = df.drop(columns=['application_name', 'author_id', 'engine_id', 'manager_id', 'status', 'decision_date',
                          'region_name', 'build_mscost', 'dpg_num_text', 'prolongation', 'update_date', 'state_id',
                          'create_date', 'annul_date', 'state_name', 'дата фин данных', 'life_cycle', 'today', 'turn', 'network', 'Unnamed: 0.1','Unnamed: 0',
                          'real_life_cycle'])
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.drop('Target 2', axis = 1)
df = df.dropna(axis=1, how='all')

non_numeric_cols = df.select_dtypes(include=['object', 'datetime', 'timedelta']).columns

label_encoder = LabelEncoder()
for col in non_numeric_cols:
    df[col] = label_encoder.fit_transform(df[col].astype(str))

In [78]:
pd.set_option('display.max_columns', None)

In [79]:
df

,gara_id,builder_id,contractor_id,designer_id,city_id,type,period_month,start_date,end_date,class,flat_area,build_qty,build_floor,build_area,com_area,park_area,flat_qty,com_qty,park_qty,flat_onerm,flat_tworm,flat_threerm,flat_fourrm,cost_plan,flat_msprice,flat_mscost,com_msprice,park_price,self_cost,unfin_cost,share_cost,warr_cost,comm_cost,cost_official,flat_inner,flat_height,dpg_date,dpg_num,dpg_add_num,bin,date,Итого краткосрочные активы,Итого краткосрочных обязательств,Денежные средства,Краткосрочная дебиторская задолженность,Запасы,Себестоимость,Доход от реализации,Итого активы,Итого капитал,Итого пассивы,Краткосрочная кредиторская задолженность,Операционная прибыль,Долгосрочные финансовые обязательства,Краткосрочные финансовые обязательства,Итоговая прибыль (убыток),Расходы на финансирование,Коэффициент текущей ликвидности,Коэффициент быстрой ликвидности,Коэффициент абсолютной ликвидности,Коэффициент оборачиваемости запасов,Коэффициент оборачиваемости активов,Коэффициент оборачиваемости кредиторской задолженности,ROA,ROE,Коэффициент чистой прибыли,Коэффициент финансового левериджа,Коэффициент финансовой независимости,Коэффициент долговой нагрузки,Коэффициент долг на капитал,Коэффициент покрытия процентов,Покрытие долга,Target,percent_of_life_cycle,season,unfin_coff
0,729352,960340000376,9.603400e+11,0.0,18178,1,13.0,146,133,4,9633.70,NaN,19,9633.70,7631.20,313.30,96,2.0,NaN,16.0,34.0,32.0,NaN,2.475277e+06,399913.089452,256938.605624,444347.877169,NaN,3.712914e+08,3.683644e+07,2.103985e+09,2.438440e+07,NaN,2.475276e+09,2,2.7,150,58,7.0,9.603400e+11,6,1.882588e+08,6.484419e+07,1.508942e+06,6.979116e+06,1.321679e+07,3.662647e+07,7.012104e+07,6.547058e+08,5.625175e+08,6.547058e+08,2.631086e+07,1.180739e+08,2.564579e+07,8.202912e+06,1.351976e+08,2.059503e+07,2.903249,0.130899,0.023270,2.771208,0.107103,1.392067,0.206501,0.240344,1.928060,1.163885,0.859191,1.0,0.060174,5.733126,3.488284,0,1.845570,2,14.881747
1,790750,960340000376,9.603400e+11,0.0,18178,1,21.0,146,158,4,19350.28,NaN,2,19350.28,16324.11,1398.43,141,12.0,NaN,26.0,35.0,40.0,NaN,3.750923e+09,362668.708918,193843.127929,362668.708918,NaN,5.626385e+08,5.400222e+08,3.188285e+09,3.210901e+07,NaN,3.750923e+09,3,2.7,156,54,9.0,9.603400e+11,6,1.862466e+08,6.415109e+07,1.492813e+06,6.904520e+06,1.307552e+07,3.623499e+07,6.937155e+07,6.477080e+08,5.565050e+08,6.477079e+08,2.602963e+07,1.168118e+08,2.537168e+07,8.115235e+06,1.337525e+08,2.037489e+07,2.903249,0.130899,0.023270,2.771208,0.107103,1.392067,0.206501,0.240344,1.928060,1.163885,0.859191,1.0,0.060174,5.733126,3.488284,0,1.140845,2,0.143970
2,845821,60340004172,6.034000e+10,0.0,18277,1,9.0,165,143,4,11723.00,NaN,19,11723.00,6650.80,1070.64,80,2.0,1635.0,8.0,40.0,24.0,NaN,2.951772e+09,469194.904792,469194.904792,491017.923619,2.182302e+06,3.243253e+08,3.243253e+08,2.877672e+09,2.877672e+07,NaN,3.201997e+09,7,2.7,159,151,1.0,6.034000e+10,6,2.467106e+07,2.262678e+07,1.325195e+05,1.450104e+07,6.312514e+06,3.073548e+07,3.384857e+07,3.057846e+07,4.390798e+06,3.057846e+07,4.291756e+06,2.245280e+06,2.989639e+06,1.091151e+06,2.118525e+06,1.079763e+05,1.090348,0.646736,0.005857,4.868977,1.106941,7.161518,0.069282,0.482492,0.062588,6.964214,0.143591,1.0,0.929396,20.794199,0.550207,0,1.854015,0,0.109875
3,781580,981040003722,9.810400e+11,0.0,18267,1,14.0,144,136,3,6698.30,NaN,16,6698.30,4842.00,NaN,60,NaN,NaN,20.0,20.0,20.0,NaN,1.358817e+09,296344.781847,268859.495725,NaN,NaN,2.038225e+08,2.044318e+08,1.154994e+09,1.154385e+07,NaN,1.358817e+09,7,2.7,152,144,NaN,9.810400e+11,6,1.456048e+06,1.023207e+06,1.306465e+04,7.499240e+04,1.332578e+06,1.113869e+06,1.188065e+06,1.478341e+06,4.551335e+05,1.478341e+06,9.336711e+05,5.164043e+04,0.000000e+00,9.306888e+03,3.984646e+04,3.877408e+01,1.423023,0.086060,0.012768,0.835875,0.803647,1.192999,0.026954,0.087549,0.033539,3.248147,0.307868,1.0,0.020449,1331.828571,5.548625,0,1.741784,2,0.150448
4,746713,80440009616,8.044001e+10,0.0,18194,1,10.0,160,138,4,4352.50,NaN,16,4352.50,2544.20,117.2

In [80]:
df.replace([np.inf, -np.inf], 0, inplace=True)
df.replace(np.NaN, 0, inplace=True)

In [81]:
X = df.drop(['Target'], axis = 1)
y = df['Target']

In [82]:
cat_cols = df.loc[: , df.nunique() < 50]


In [83]:
cat_cols

,city_id,type,period_month,class,build_qty,build_floor,com_qty,flat_fourrm,flat_inner,flat_height,dpg_add_num,date,Коэффициент долговой нагрузки,Коэффициент покрытия процентов,Target,season
0,18178,1,13.0,4,0.0,19,2.0,0.0,2,2.7,7.0,6,1.0,5.733126,0,2
1,18178,1,21.0,4,0.0,2,12.0,0.0,3,2.7,9.0,6,1.0,5.733126,0,2
2,18277,1,9.0,4,0.0,19,2.0,0.0,7,2.7,1.0,6,1.0,20.794199,0,0
3,18267,1,14.0,3,0.0,16,0.0,0.0,7,2.7,0.0,6,1.0,1331.828571,0,2
4,18194,1,10.0,4,0.0,16,3.0,0.0,4,2.7,1.0,6,1.0,0.000000,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,18178,1,12.0,4,0.0,4,29.0,0.0,4,2.7,0.0,7,1.0,26.373242,0,1
207,18277,1,10.0,3,0.0,16,0.0,0.0,3,3.0,0.0,7,1.0,0.000000,0,1
208,18194,1,23.0,4,0.0,19,0.0,0.0,4,2.7,0.0,7,1.0,0.000000,0,1
209,18274,1,10.0,4,0.0,16,0.0,0.0,4,2.7,0.0,7,1.0,0.000000,0,1


In [84]:
iv_score_dict = {}
for col in tqdm(X.columns):
  if col in cat_cols:
    optb = optbinning.OptimalBinning(dtype='categorical')
    optb.fit(df[col], df['Target'])
  else:
    optb = optbinning.OptimalBinning(dtype='numerical')
    optb.fit(df[col], df['Target'])
  binning_table = optb.binning_table
  binning_table.build()
  iv_score_dict[col] = binning_table.iv
iv_score_df = pd.Series(iv_score_dict)
iv_score_df.sort_values(ascending=False, inplace=True)

100%|██████████| 75/75 [00:04<00:00, 15.21it/s]


In [85]:
iv_score_df

,0
Коэффициент покрытия процентов,2.553221
Коэффициент оборачиваемости активов,1.297831
period_month,1.172679
city_id,0.947601
share_cost,0.938570
...,...
park_price,0.083468
dpg_add_num,0.082223
class,0.021413
type,0.000000


In [86]:
iv_score_df2 = pd.DataFrame(iv_score_df, columns=['Numbers']).reset_index()
iv_score_df3 = iv_score_df2[iv_score_df2['Numbers'].between(0.02, 3)]
selected_features = iv_score_df3['index']
selected_features

,index
0,Коэффициент покрытия процентов
1,Коэффициент оборачиваемости активов
2,period_month
3,city_id
4,share_cost
...,...
68,gara_id
69,Коэффициент долг на капитал
70,park_price
71,dpg_add_num


In [87]:
df2 = pd.concat([df[selected_features], df['Target']], axis = 1)


In [88]:
X = df2.drop(['Target'], axis = 1)
y = df2['Target']

In [62]:
binning_process = optbinning.BinningProcess(
  variable_names=X.columns.to_list())
# estimator
estimator = linear_model.LogisticRegression()
# scorecard
scorecard = optbinning.Scorecard(
  binning_process=binning_process,
  estimator=estimator,
  scaling_method="min_max",
  scaling_method_params={"min": 300, "max": 850},
# scaling_method = "pdo_odds",
# scaling_method_params = {"pdo": 20, "odds": 50, "scorecard_points": 100},
#intercept_based=True,
#reverse_scorecard=True
)

In [63]:
scorecard.fit(X, y)


Scorecard(binning_process=BinningProcess(variable_names=['Коэффициент покрытия '
                                                         'процентов',
                                                         'Коэффициент '
                                                         'оборачиваемости '
                                                         'активов',
                                                         'period_month',
                                                         'city_id',
                                                         'share_cost',
                                                         'com_qty',
                                                         'contractor_id',
                                                         'dpg_date',
                                                         'percent_of_life_cycle',
                                                         'build_floor',
                                                         'Итого краткосрочные '
                                                         'активы',
                                                         'Итого пассивы',
                                                         'Итого активы',
                                                         'Расходы на '
                                                         'финансирование',
                                                         'Краткосрочная '
                                                         'кредиторская '
                                                         'задолженность',
                                                         'ROE',
                                                         'Операционная прибыль',
                                                         'date', 'ROA',
                                                         'Коэффициент долговой '
                                                         'нагрузки',
                                                         'end_date', 'park_qty',
                                                         'Итоговая прибыль '
                                                         '(убыток)',
                                                         'unfin_coff',
                                                         'start_date',
                                                         'com_msprice',
                                                         'dpg_num',
                                                         'Себестоимость',
                                                         'flat_threerm',
                                                         'designer_id', ...]),
          estimator=LogisticRegression(), scaling_method='min_max',
          scaling_method_params={'max': 850, 'min': 300})

In [64]:
scorecard.intercept_


0

In [65]:
scorecard_df = scorecard.table(style="detailed")


In [66]:
prediction = scorecard.predict_proba(X)[:, 1]


In [1]:
prediction

NameError: name 'prediction' is not defined

In [67]:
scorecard_df

,Variable,Bin id,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS,Coefficient,Points
0,Коэффициент покрытия процентов,0,"(-inf, -4.35)",11,0.052133,8,3,0.272727,-0.855882,0.050918,0.006177,0.236078,9.391859
1,Коэффициент покрытия процентов,1,"[-4.35, 0.09)",131,0.620853,118,13,0.099237,0.369024,0.073833,0.009177,0.236078,6.420166
2,Коэффициент покрытия процентов,2,"[0.09, 1.47)",22,0.104265,21,1,0.045455,1.207812,0.097714,0.011522,0.236078,4.385219
3,Коэффициент покрытия процентов,3,"[1.47, 11.36)",26,0.123223,20,6,0.230769,-0.632738,0.061380,0.007547,0.236078,8.850500
4,Коэффициент покрытия процентов,4,"[11.36, inf)",21,0.099526,15,6,0.285714,-0.920420,0.114573,0.013837,0.236078,9.548433
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4,dpg_add_num,4,Missing,0,0.000000,0,0,0.000000,0.000000,0.000000,0.000000,-0.364289,7.315440
0,class,0,"(-inf, 3.50)",27,0.127962,22,5,0.185185,-0.355106,0.018300,0.002276,0.240209,8.192026
1,class,1,"[3.50, inf)",184,0.872038,160,24,0.130435,0.060409,0.003113,0.000389,0.240209,7.166320
2,class,2,Special,0,0.000000,0,0,0.000000,0.000000,0.000000,0.000000,0.240209,7.315440


In [68]:
scorecard_df.to_excel('scorecard.xlsx')

In [69]:
print(f'Logit ROC_AUC: {roc_auc_score(y, prediction )}')

Logit ROC_AUC: 1.0


In [70]:
scorecard_df2 = scorecard_df.drop_duplicates(subset=['Variable'])
scorecard_df2 = scorecard_df2[['Variable', 'Coefficient']]
scorecard_df2

,Variable,Coefficient
0,Коэффициент покрытия процентов,0.236078
0,Коэффициент оборачиваемости активов,-0.194104
0,period_month,-0.408693
0,city_id,-0.128247
0,share_cost,-1.112613
...,...,...
0,gara_id,-0.802014
0,Коэффициент долг на капитал,-0.043354
0,park_price,-0.140915
0,dpg_add_num,-0.364289


In [71]:
xx = (X.loc[:,~X.columns.isin(['debt_to_EBIT', 'debt_to_equity',
                                'debt_ratio', 'debt_to_sales',
                                'leverage', 'nca_to_net_worth',
                                'interest_expense_to_debt',
                                'interest_coverage_ratio'])]).copy()

In [72]:
plt.figure(figsize=(100, 50))
# Define the mask to set the values in the upper triangle to True
mask = np.triu(np.ones_like(xx.corr(), dtype=bool))
heatmap = sns.heatmap(xx.corr(), mask=mask, vmin=-1, vmax=1, annot=True, cmap='BrBG')
heatmap.set_title('Triangle Correlation Heatmap', fontdict={'fontsize':18}, pad=16)

Output hidden; open in https://colab.research.google.com to view.